In [1]:
import numpy as np
import pandas as pd
from scipy.io import loadmat

import pandas as pd
import time
import os
import sys
import zarr
import napari 
import dask.array as da 

pythonPackagePath = os.path.abspath(r'C:\Users\Lab admin\Desktop\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)
from parallel import Detector
from gaussian_visualization import visualize_3D_gaussians

In [2]:
base_dir =  r'C:\Users\Lab admin\Desktop\u-track3D\testTrackability'

# Define the file directory and name
input_file_directory = 'lumenoid_1full_analysis/'
# zarr_file_directory = input_file_directory + 'zarr_file/all_channels_data'

# zarr_full_path = os.path.join(base_dir, zarr_file_directory)

In [3]:
# Load the tracks
# path_to_detections = os.path.join(base_dir, input_file_directory) + 'output/uTrack3DPackage/detect3D/channel_3.mat'
path_to_tracks = os.path.join(base_dir, input_file_directory) + 'tracks/Channel_1_tracking_result.mat'
# path_to_tracks = os.path.join(base_dir, input_file_directory) + 'tracking_results/12/Channel_1_tracking_result_12.mat'
tracks_dict = loadmat(path_to_tracks)

In [4]:
# Get the keys of the dictionary
# tracks_dict.keys()

In [4]:
# Get the key for your detections
key = 'tracksFinal'  # Replace with your actual key name
# track_data = tracks_dict[key]

In [22]:
def convert_tracks_to_dataframe(tracks_dict, key):
    """
    Convert MATLAB tracking data to a pandas DataFrame.
    
    Parameters:
    tracks_dict : dict
        Dictionary containing the MATLAB data (loaded using scipy.io.loadmat)
    key : str
        Key for the specific tracking data in the dictionary
    
    Returns:
    pd.DataFrame: DataFrame with columns [frame, mu_x, mu_y, mu_z, track_id]
    """
    import numpy as np
    import pandas as pd
    import scipy.io as sio
    
    # # Load the MATLAB file
    # mat_data = sio.loadmat(filepath, squeeze_me=False, struct_as_record=False)
    
    # Extract the tracks structure
    tracks = tracks_dict[key].flatten()
    
    # Create an empty list to store all track data
    all_tracks = []
    regular_tracks = []
    split_merge_tracks = []
    
    # Process each track
    for track_id in range(len(tracks)):

        # Handle numpy.void objects by accessing elements with field names
        track = tracks[track_id]
        # Access fields using dictionary-like indexing for numpy.void objects
        track_coords = track['tracksCoordAmpCG']
        seq_events = track['seqOfEvents']
        
        # Handle different shapes of seqOfEvents
        if seq_events.size == 0:
            continue  # Skip empty tracks
            
        # Reshape if necessary to ensure consistent format
        if len(seq_events.shape) == 1:
            seq_events = seq_events.reshape(1, -1)

        if np.isnan(seq_events[:, -1]).all():

            track_coords = track_coords[0]

            # save the track as a regular track
            regular_tracks.append(track_id)

            # Find start and end frames
            start_frame = int(seq_events[0, 0])
            end_frame = int(seq_events[-1, 0])

            # Determine the total number of frames from the size of tracksCoordAmpCG
            # Each frame has 8 columns [x y z a dx dy dz da]
            num_cols = track_coords.shape[0]
            num_frames = num_cols // 8

            # For each frame in the track's lifespan
            for frame_idx in range(num_frames):
                frame_number = start_frame + frame_idx
                # Extract x, y, z coordinates for current frame
                col_idx = frame_idx * 8

                x = track_coords[col_idx]
                y = track_coords[col_idx + 1]
                z = track_coords[col_idx + 2]
                amplitude = track_coords[col_idx + 3]

                track_data = {
                    'frame': frame_number,
                    'mu_x': round(x) if not np.isnan(x) else None,
                    'mu_y': round(y) if not np.isnan(y) else None,
                    'mu_z': round(z) if not np.isnan(z) else None,
                    'amplitude': amplitude,
                    'track_id': track_id  
                }

                all_tracks.append(track_data)

        else:
            # save the track as a split/merge track
            split_merge_tracks.append(track_id)

            # Determine if the track is split or merged (this seems a bit rudimentary, probably could be better)
            # Abhishek Raghunathan, 04/14/25
            if not np.isnan(seq_events[1, -1]): # This is assuming that we have only a single split event, it will fail otherwise.
                # Also that all split events are position 1 in seq_events
                track_flag = 'split'
                # print(f'Track {track_id} is split.')

            if not np.isnan(seq_events[2, -1]): # This is assuming that we have only a single merge event, it will fail otherwise.
                # Also that all merge events are position 2 in seq_events
                track_flag = 'merge'
                # print(f'Track {track_id} is merged.')
            

            # Process each segment in the track
            segments = np.unique(seq_events[:, 2]).astype(int)
            segments_min = np.min(segments) # Assuming the lowest segment ID is the first one (might not be true).
            start_frame_original = [] #To store the original start frame
            
            for segment_id in segments:
                # Find events related to this segment
                segment_events = seq_events[seq_events[:, 2] == segment_id]
                
                # Get start and end frames for this segment
                start_events = segment_events[segment_events[:, 1] == 1]
                end_events = segment_events[segment_events[:, 1] == 2]

                if segment_id == segments_min: # Assuming the lowest segment ID has the track which started first (lower value of first frame). CHECK THIS.
                    start_frame_original.append(int(start_events[0,0])) # This is the original start frame for the split or merge track
                    # print(start_frame_original)
                
                if start_events.size > 0 and end_events.size > 0:
                    start_frame = int(start_events[0, 0])
                    end_frame = int(end_events[0, 0])
                    
                    # Get row index for this segment (0-indexed)
                    segment_idx = segment_id - 1

                    # Get the row data for this segment
                    if segment_idx < len(track_coords):
                        segment_data = track_coords[segment_idx]
                        
                        # Calculate number of frames in this segment
                        segment_frames = end_frame - start_frame + 1
                        
                        # Process each frame in this segment
                        for frame_offset in range(segment_frames):
                            frame_number = start_frame + frame_offset

                            if (track_flag == 'split') or (track_flag == 'merge'): # This flag is unnecessary here, have it for legacy reasons.
                                col_idx = (start_frame + frame_offset - start_frame_original[0]) * 8 # This will handle cases where start_frame_original is not 1
                            else:
                                col_idx = frame_offset * 8
                            
                            # if track_id == 4370:
                            #     print(col_idx)
                            
                            
                            # Check if indices are within bounds
                            if col_idx + 3 < len(segment_data):
                                x = segment_data[col_idx]
                                y = segment_data[col_idx + 1]
                                z = segment_data[col_idx + 2]
                                amplitude = segment_data[col_idx + 3]
                                
                                track_data = {
                                    'frame': frame_number,
                                    'mu_x': round(x) if not np.isnan(x) else None,
                                    'mu_y': round(y) if not np.isnan(y) else None,
                                    'mu_z': round(z) if not np.isnan(z) else None,
                                    'amplitude': amplitude,
                                    'track_id': track_id,
                                    'segment_id': segment_id
                                }
                                all_tracks.append(track_data)
        
    
    track_df = pd.DataFrame(all_tracks)
    
    # Sort by track_id and frame
    track_df = track_df.sort_values(['track_id', 'frame'])

    return track_df, regular_tracks, split_merge_tracks

In [23]:
df, regular_tracks, split_merge_tracks = convert_tracks_to_dataframe(tracks_dict, key)

In [24]:
# # Save df in path_to_tracks as a pickle file
output_path = path_to_tracks.replace('.mat', '.pkl')
df.to_pickle(output_path)

In [ ]:
len(split_merge_tracks)
# split_merge_tracks

,frame,mu_x,mu_y,mu_z,amplitude,track_id
0,1,101.0,178.0,72.0,658.0,0
1,2,100.0,177.0,71.0,858.0,0
2,3,101.0,177.0,71.0,663.0,0
3,4,101.0,178.0,72.0,685.0,0
4,5,101.0,178.0,71.0,712.0,0
5,6,102.0,178.0,70.0,586.0,0
6,7,101.0,178.0,70.0,592.0,0
7,8,102.0,178.0,70.0,596.0,0
8,9,102.0,177.0,70.0,591.0,0
9,10,101.0,178.0,70.0,535.0,0


In [29]:
# df[(df['track_id'] == 37)]
# df[(df['track_id'] == 37) & (df['segment_id'] == 1.0)]

In [ ]:
# tracks = tracks_dict[key].flatten()
# track = tracks[37]
# track_coords = track['tracksCoordAmpCG']
# seq_events = track['seqOfEvents']
# seq_events

array([[ 1.,  1.,  1., nan],
       [ 9.,  2.,  1., nan]])

In [ ]:
# track_coords[0]

array([ 67.59114249, 216.        ,  28.22528883, 332.        ,
         0.5       ,   0.5       ,   0.5       ,   0.5       ,
        67.5666356 , 216.41332712,  28.57921715, 340.        ,
         0.5       ,   0.5       ,   0.5       ,   0.5       ,
        67.49072876, 216.52134541,  28.49460975, 320.        ,
         0.5       ,   0.5       ,   0.5       ,   0.5       ,
        68.        , 217.        ,  28.        , 257.        ,
         0.5       ,   0.5       ,   0.5       ,   0.5       ,
        67.33333333, 217.        ,  27.32899023, 311.        ,
         0.5       ,   0.5       ,   0.5       ,   0.5       ,
        67.64664311, 217.        ,  26.68669022, 300.        ,
         0.5       ,   0.5       ,   0.5       ,   0.5       ,
        66.        , 217.        ,  28.50698603, 254.        ,
         0.5       ,   0.5       ,   0.5       ,   0.5       ,
        67.        , 216.65761511,  26.33884298, 290.        ,
         0.5       ,   0.5       ,   0.5       ,   0.5 

In [28]:
# track_df = df[df['track_id'] == 33789]
# #sort based on segment id values
# track_df.sort_values(['segment_id', 'frame'])

In [14]:
# # get non NaN values in track_coords[1]
# track_coords[1][~np.isnan(track_coords[1])]

In [15]:
# # Subset rows of df with NaNs in any of mu_x, mu_y, mu_z
# df_nan = df[df[['mu_x', 'mu_y', 'mu_z']].isnull().any(axis=1)]
# # Get the track IDs of these rows
# nan_track_ids = df_nan['track_id'].unique()
# # Count the number of times each track ID appears in df_nan
# track_id_counts = df_nan['track_id'].value_counts()
# # Filter track IDs that appear more than once
# track_id_counts = track_id_counts[track_id_counts > 1]



In [16]:
# track_id_counts